In [ ]:
# ✅ Install library jika diperlukan
# !pip install tensorflow tensorflow-datasets tensorflow-addons tensorflow-hub matplotlib

import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import tensorflow_hub as hub
from collections import Counter

print("TensorFlow version:", tf.__version__)

##############################################
# 1️⃣ Download & Tokenize Shakespeare Dataset
##############################################
try:
    shakespeare_url = "https://homl.info/shakespeare"
    filepath = keras.utils.get_file("shakespeare.txt", shakespeare_url)
    with open(filepath, encoding='utf-8') as f:
        shakespeare_text = f.read()

    print(f"Shakespeare text loaded: {len(shakespeare_text)} characters")

    # Tokenizer untuk karakter
    tokenizer = keras.preprocessing.text.Tokenizer(char_level=True)
    tokenizer.fit_on_texts([shakespeare_text])

    max_id = len(tokenizer.word_index)
    print(f"Vocabulary size: {max_id}")

    # Encode text
    [encoded] = np.array(tokenizer.texts_to_sequences([shakespeare_text])) - 1
    print(f"Encoded text shape: {encoded.shape}")

except Exception as e:
    print(f"Error in Shakespeare dataset preparation: {e}")

##############################################
# 2️⃣ Preparing Dataset for Training
##############################################
try:
    # Fix: Gunakan len(encoded) sebagai dataset_size
    dataset_size = len(encoded)
    train_size = dataset_size * 90 // 100

    window_length = 100 + 1
    batch_size = 32
    n_steps = 100

    # Buat dataset untuk training
    dataset = tf.data.Dataset.from_tensor_slices(encoded[:train_size])
    dataset = dataset.window(window_length, shift=1, drop_remainder=True)
    dataset = dataset.flat_map(lambda window: window.batch(window_length))
    dataset = dataset.shuffle(10000).batch(batch_size)
    dataset = dataset.map(lambda windows: (windows[:, :-1], windows[:, 1:]))
    dataset = dataset.map(lambda X, Y: (tf.one_hot(X, depth=max_id), Y))
    dataset = dataset.prefetch(1)

    print("Dataset prepared successfully")

except Exception as e:
    print(f"Error in dataset preparation: {e}")

##############################################
# 3️⃣ Building Char-RNN Model (2 GRU Layers)
##############################################
try:
    model = keras.models.Sequential([
        keras.layers.GRU(128, return_sequences=True, input_shape=[None, max_id]),
        keras.layers.GRU(128, return_sequences=True),
        keras.layers.TimeDistributed(keras.layers.Dense(max_id, activation="softmax"))
    ])

    model.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
    print("Model compiled successfully")

    # Reduce epochs untuk testing
    print("Starting training...")
    history = model.fit(dataset, epochs=5, verbose=1)  # Reduced from 20 to 5

except Exception as e:
    print(f"Error in model training: {e}")

##############################################
# 4️⃣ Generating Shakespeare Text
##############################################
def preprocess(texts):
    X = np.array(tokenizer.texts_to_sequences(texts)) - 1
    return tf.one_hot(X, max_id)

def next_char(text, temperature=1.0):
    X_new = preprocess([text])
    y_proba = model.predict(X_new, verbose=0)[0, -1:]
    rescaled_logits = tf.math.log(y_proba) / temperature
    char_id = tf.random.categorical(rescaled_logits, num_samples=1) + 1
    return tokenizer.sequences_to_texts(char_id.numpy())[0]

def complete_text(text, n_chars=50, temperature=1.0):
    for _ in range(n_chars):
        text += next_char(text, temperature)
    return text

try:
    generated_text = complete_text("To be", temperature=0.7)
    print(f"Generated text: {generated_text}")
except Exception as e:
    print(f"Error in text generation: {e}")

##############################################
# 5️⃣ Sentiment Analysis using IMDb Dataset
##############################################
try:
    print("Loading IMDb dataset...")
    datasets, info = tfds.load("imdb_reviews", as_supervised=True, with_info=True)
    train_size = info.splits["train"].num_examples
    print(f"IMDb dataset loaded. Train size: {train_size}")

    def preprocess_imdb(X_batch, y_batch):
        X_batch = tf.strings.substr(X_batch, 0, 300)
        X_batch = tf.strings.regex_replace(X_batch, b"<br\\s*/?>", b" ")
        X_batch = tf.strings.regex_replace(X_batch, b"[^a-zA-Z']", b" ")
        X_batch = tf.strings.split(X_batch)
        return X_batch.to_tensor(default_value=b"<pad>"), y_batch

    # Build vocabulary
    print("Building vocabulary...")
    vocabulary = Counter()
    for X_batch, y_batch in datasets["train"].batch(32).map(preprocess_imdb).take(100):  # Limit for faster processing
        for review in X_batch:
            vocabulary.update(list(review.numpy()))

    vocab_size = 10000
    truncated_vocabulary = [word for word, count in vocabulary.most_common()[:vocab_size]]
    words = tf.constant(truncated_vocabulary)
    word_ids = tf.range(len(truncated_vocabulary), dtype=tf.int64)
    vocab_init = tf.lookup.KeyValueTensorInitializer(words, word_ids)
    num_oov_buckets = 1000
    table = tf.lookup.StaticVocabularyTable(vocab_init, num_oov_buckets)

    def encode_words(X_batch, y_batch):
        return table.lookup(X_batch), y_batch

    train_set = datasets["train"].batch(32).map(preprocess_imdb).map(encode_words).take(100).prefetch(1)

    embed_size = 128
    sentiment_model = keras.models.Sequential([
        keras.layers.Embedding(vocab_size + num_oov_buckets, embed_size, input_shape=[None]),
        keras.layers.GRU(128, return_sequences=True),
        keras.layers.GRU(128),
        keras.layers.Dense(1, activation="sigmoid")
    ])

    sentiment_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
    print("Training sentiment model...")
    history = sentiment_model.fit(train_set, epochs=2, verbose=1)  # Reduced epochs

except Exception as e:
    print(f"Error in sentiment analysis: {e}")

##############################################
# 6️⃣ Transfer Learning with Pretrained Embeddings
##############################################
try:
    print("Setting up transfer learning model...")

    # Preprocess data for hub model
    def preprocess_for_hub(X_batch, y_batch):
        X_batch = tf.strings.substr(X_batch, 0, 300)
        X_batch = tf.strings.regex_replace(X_batch, b"<br\\s*/?>", b" ")
        X_batch = tf.strings.regex_replace(X_batch, b"[^a-zA-Z']", b" ")
        return X_batch, y_batch

    hub_model = keras.Sequential([
        hub.KerasLayer("https://tfhub.dev/google/tf2-preview/nnlm-en-dim50/1",
                       dtype=tf.string, input_shape=[], output_shape=[50]),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid")
    ])

    hub_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

    # Prepare dataset for hub model
    hub_train_set = datasets["train"].batch(32).map(preprocess_for_hub).take(50).prefetch(1)

    print("Training transfer learning model...")
    history = hub_model.fit(hub_train_set, epochs=2, verbose=1)  # Reduced epochs

except Exception as e:
    print(f"Error in transfer learning: {e}")

print("All sections completed!")

TensorFlow version: 2.18.0
Shakespeare text loaded: 1115394 characters
Vocabulary size: 39
Encoded text shape: (1115394,)
Dataset prepared successfully
Model compiled successfully
Starting training...
Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


31368/31368 ━━━━━━━━━━━━━━━━━━━━ 10560s 336ms/step - loss: 1.0818
Epoch 2/5


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


31368/31368 ━━━━━━━━━━━━━━━━━━━━ 10354s 330ms/step - loss: 0.9359
Epoch 3/5


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


 7141/31368 ━━━━━━━━━━━━━━━━━━━━ 2:14:35 333ms/step - loss: 1.0193

# 📚 Chapter 16 - Natural Language Processing with RNNs and Attention

---

## 🔸 1. Apa Itu NLP?
**Natural Language Processing (NLP)** adalah bidang machine learning yang memproses dan menganalisis data berbasis teks dan bahasa.

Contoh aplikasi NLP:
- Machine Translation
- Sentiment Analysis
- Text Generation
- Question Answering

---

## 🔸 2. Representasi Teks → Tokenisasi
- **Character-level Tokenization** → tiap karakter diubah menjadi angka
- **Word-level Tokenization** → tiap kata diubah menjadi angka (lebih umum)
- **Subword Tokenization (BPE, SentencePiece)** → kombinasi keduanya → dipakai oleh BERT, GPT

---

## 🔸 3. Model RNN untuk Teks
- **Char-RNN** → belajar urutan karakter → cocok untuk text generation (misalnya Shakespeare)
- **Embedding Layer** → mengubah integer token → representasi vektor → memperbaiki representasi makna kata.

---

## 🔸 4. Text Generation
1. Prediksi token berikutnya
2. Sampling → **temperature** mengontrol kreativitas output:
   - Temperature rendah → teks lebih konservatif
   - Temperature tinggi → teks lebih bervariasi dan kreatif

---

## 🔸 5. Sentiment Analysis Example
✅ **IMDb Dataset** → binary sentiment (positive/negative)
Langkah umum:
1. Tokenisasi + Cleaning
2. Encoding → token integer
3. **Embedding → RNN → Dense output**

---

## 🔸 6. Pretrained Embeddings
**Transfer Learning dalam NLP** → gunakan embedding yang sudah dilatih:
- **TensorFlow Hub** menyediakan pretrained models
- Contoh: `https://tfhub.dev/google/tf2-preview/nnlm-en-dim50/1`

---

## 🔸 7. Attention Mechanism (Pendahuluan)
**Attention** → memungkinkan model untuk **fokus** pada bagian input yang relevan → penting untuk:
- **Machine Translation**
- **Summarization**
- **Transformers (Chapter 16 lanjutan & Chapter 17)**

---

## ✅ Kesimpulan
- ✅ RNN & LSTM → dasar untuk sequence modeling di NLP
- ✅ Transfer Learning (pretrained embeddings) → percepat training + hasil lebih baik
- ✅ Attention → fundamental untuk NLP modern → lanjut ke Transformers
